# Unified Multi-Task RoBERTa AES (10 Traits)

Custom multi-head architecture with mean-pooling. **BERTScore F1 is concatenated to the pooled embedding** before the per-trait heads.

## 0. Imports, Seed, Device

In [1]:
!pip install bert_score
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import cohen_kappa_score
from tqdm import tqdm
from bert_score import score
import gc
import random
import itertools
import warnings
import os

warnings.filterwarnings("ignore")

def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        if hasattr(torch, 'xpu') and torch.xpu.is_available():
            torch.xpu.manual_seed(seed)
            torch.xpu.manual_seed_all(seed)
    except ImportError:
        print("Warning: intel_extension_for_pytorch (ipex) tidak ditemukan.")
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(42)

def get_device():
    if hasattr(torch, 'xpu') and torch.xpu.is_available():
        return torch.device("xpu")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = get_device()
print(f"Using Device: {DEVICE}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.0 MB/s eta 0:00:00
Using Device: cuda


## 1. Konfigurasi Eksperimen

In [2]:
TEXT_TRAITS = [
    "organizational_structure", "coherence", "essay_length",
    "grammatical_accuracy", "grammatical_diversity",
    "lexical_accuracy", "lexical_diversity", "punctuation_accuracy", "argument_clarity", "justifying_persuasiveness"
]
TRAIT_COLS = {
    "organizational_structure": "organizational_structure(ground_truth)",
    "coherence":                "coherence(ground_truth)",
    "essay_length":             "essay_length(ground_truth)",
    "grammatical_accuracy":     "grammatical_accuracy(ground_truth)",
    "grammatical_diversity":    "grammatical_diversity(ground_truth)",
    "lexical_accuracy":         "lexical_accuracy(ground_truth)",
    "lexical_diversity":        "lexical_diversity(ground_truth)",
    "punctuation_accuracy":     "punctuation_accuracy(ground_truth)",
    "argument_clarity":         "argument_clarity(ground_truth)",
    "justifying_persuasiveness":"justifying_persuasiveness(ground_truth)"
}

GROUP_COL            = "graph"
MODEL_NAME           = "roberta-base"
MAX_LEN              = 512
BATCH_SIZE           = 2
ACCUMULATION_STEPS   = 4              # Effective batch size = 8
UNFREEZE_LAYERS      = 6
DATA_FILE            = "short_gemma_bertscore.csv"
BEST_MODEL_SAVE_PATH = "best_roberta_unified_final_2.pth"
ID_COL               = 'Essay'
K_FOLDS              = 3

CHECKPOINT_DIR = "checkpoints_unified"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Multi-Task Traits (n=10): {TEXT_TRAITS}")
print(f"Checkpoint Dir    : {CHECKPOINT_DIR}/")

Multi-Task Traits (n=10): ['organizational_structure', 'coherence', 'essay_length', 'grammatical_accuracy', 'grammatical_diversity', 'lexical_accuracy', 'lexical_diversity', 'punctuation_accuracy', 'argument_clarity', 'justifying_persuasiveness']
Checkpoint Dir    : checkpoints_unified/


## 2. Prapemrosesan & BERTScore

In [3]:
def prepare_data(csv_path):
    df = pd.read_csv(csv_path).dropna(subset=['description']).reset_index(drop=True)
    if 'bertscore_f1' not in df.columns:
        print("\n[*] Menghitung BERTScore antara Esai dan Description...")
        cands = df['Essay'].fillna("").tolist()
        refs  = df['description'].fillna("").tolist()
        P, R, F1 = score(cands, refs, lang="en", verbose=True)
        df['bertscore_precision'] = P.numpy()
        df['bertscore_recall']    = R.numpy()
        df['bertscore_f1']        = F1.numpy()
        df.to_csv(csv_path, index=False)
        print(f"[*] Selesai! Data disimpan ke {csv_path}\n")
    return df

## 3. Dataset (Multi-Task, with BERTScore)

In [4]:
class MultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, traits):
        self.df         = df.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_len    = max_len
        self.traits     = traits
        self.questions  = self.df['Question'].fillna("").values
        self.deplots    = self.df['description'].fillna("").values
        self.essays     = self.df['Essay'].fillna("").values
        self.bertscores = self.df['bertscore_f1'].fillna(0.0).values

        self.targets = np.stack(
            [self.df[TRAIT_COLS[t]].values.astype(np.float32) for t in traits],
            axis=1,
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        suffix_text   = f"\n\nGraph Data: {self.deplots[index]} \n\nQuestion: {self.questions[index]}"
        suffix_tokens = self.tokenizer(suffix_text, add_special_tokens=False)['input_ids']

        prefix_text   = f"Essay: {self.essays[index]}"
        prefix_tokens = self.tokenizer(prefix_text, add_special_tokens=False)['input_ids']

        num_special_tokens = 2
        max_suffix_len = self.max_len - num_special_tokens
        if len(suffix_tokens) > max_suffix_len:
            suffix_tokens = suffix_tokens[:max_suffix_len]

        budget = self.max_len - len(suffix_tokens) - num_special_tokens
        prefix_tokens = prefix_tokens[:budget] if budget > 0 else []

        combined_tokens = prefix_tokens + suffix_tokens
        input_ids       = [self.tokenizer.cls_token_id] + combined_tokens + [self.tokenizer.sep_token_id]
        attention_mask  = [1] * len(input_ids)
        padding_length  = self.max_len - len(input_ids)
        if padding_length > 0:
            input_ids      += [self.tokenizer.pad_token_id] * padding_length
            attention_mask += [0] * padding_length

        return {
            'input_ids':      torch.tensor(input_ids,      dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'bertscore':      torch.tensor(self.bertscores[index], dtype=torch.float),
            'targets':        torch.tensor(self.targets[index],    dtype=torch.float),
        }

## 4. Model: RobertaSemanticAES (Multi-Head + BERTScore Concat)

Pooled embedding `[B, 768]` is concatenated with BERTScore `[B, 1]` -> `[B, 769]`, then routed to per-trait MLP heads.

In [5]:
class RobertaSemanticAES(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout_rate=0.1, hidden_dim=256):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)

        for p in self.roberta.parameters():
            p.requires_grad = False
        for layer in self.roberta.encoder.layer[-UNFREEZE_LAYERS:]:
            for p in layer.parameters():
                p.requires_grad = True

        self.text_dim    = self.roberta.config.hidden_size  # 768
        self.feature_dim = self.text_dim + 1                 # + bertscore
        self.dropout     = nn.Dropout(dropout_rate)

        self.txt_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.feature_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dim, 1),
            ) for _ in range(len(TEXT_TRAITS))
        ])

    def cls_pooling(self, model_output):
        # CLS = first token embedding
        return model_output.last_hidden_state[:, 0, :]

    def forward(self, input_ids, attention_mask, bertscore):
        txt_out    = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        bert_embed = self.cls_pooling(txt_out)  # [B, 768]
        bert_embed = self.dropout(bert_embed)
        combined   = torch.cat((bert_embed, bertscore), dim=1)  # [B, 769]
        txt_preds  = [h(combined) for h in self.txt_heads]
        return torch.cat(txt_preds, dim=1)  # [B, n_traits]

## 5. Helpers: QWK Eval & Checkpointing

In [6]:
def evaluate_qwk(model, dataloader):
    model.eval()
    all_preds   = [[] for _ in TEXT_TRAITS]
    all_targets = [[] for _ in TEXT_TRAITS]
    with torch.no_grad():
        for b in dataloader:
            ids  = b['input_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            bs   = b['bertscore'].to(DEVICE).unsqueeze(1)
            tgt  = b['targets'].to(DEVICE)
            if DEVICE.type in ["cuda", "xpu"]:
                dtype = torch.bfloat16 if DEVICE.type == "xpu" else torch.float16
                with torch.autocast(device_type=DEVICE.type, dtype=dtype):
                    pred = model(input_ids=ids, attention_mask=mask, bertscore=bs)
            else:
                pred = model(input_ids=ids, attention_mask=mask, bertscore=bs)
            pred = pred.float().cpu().numpy()
            tgt  = tgt.float().cpu().numpy()
            for ti in range(len(TEXT_TRAITS)):
                all_preds[ti].extend(pred[:, ti].flatten())
                all_targets[ti].extend(tgt[:, ti].flatten())

    qwks = {}
    for ti, trait in enumerate(TEXT_TRAITS):
        pred_classes = np.round(np.array(all_preds[ti])   * 2).astype(int)
        true_classes = np.round(np.array(all_targets[ti]) * 2).astype(int)
        qwks[trait]  = cohen_kappa_score(true_classes, pred_classes, weights='quadratic')
    qwks['mean'] = float(np.mean([qwks[t] for t in TEXT_TRAITS]))
    return qwks


def save_checkpoint(path, epoch, model, optimizer, scaler, extra=None):
    state = {
        'epoch':           epoch,
        'model_state':     model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state':    scaler.state_dict() if scaler else None,
    }
    if extra:
        state.update(extra)
    torch.save(state, path)
    print(f"   Checkpoint tersimpan: {path} (epoch {epoch+1})")


def load_checkpoint(path, model, optimizer=None, scaler=None):
    state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state['model_state'])
    if optimizer and state.get('optimizer_state'):
        optimizer.load_state_dict(state['optimizer_state'])
    if scaler and state.get('scaler_state'):
        scaler.load_state_dict(state['scaler_state'])
    epoch = state.get('epoch', -1)
    extra = {k: v for k, v in state.items()
             if k not in ('epoch', 'model_state', 'optimizer_state', 'scaler_state')}
    print(f"   Checkpoint dimuat dari: {path} (resume epoch {epoch+1})")
    return epoch, extra


def _clear_gpu():
    gc.collect()
    if DEVICE.type == "xpu":    torch.xpu.empty_cache()
    elif DEVICE.type == "cuda": torch.cuda.empty_cache()

## STAGE 1 — Memuat / Membagi Data

In [7]:
df        = prepare_data(DATA_FILE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if os.path.exists('train_df_gemma.csv') and os.path.exists('test_df_gemma.csv'):
    print("Memuat train_gemma.csv & test_gemma.csv ...")
    df_train = pd.read_csv('train_df_gemma.csv')
    df_test  = pd.read_csv('test_df_gemma.csv')

elif os.path.exists('../../train_df.csv') and os.path.exists('../../test_df.csv'):
    print("Memuat train_df.csv & test_df.csv lalu mapping ke Gemma data...")
    df_train_ref = pd.read_csv('../../train_df.csv')
    df_test_ref  = pd.read_csv('../../test_df.csv')
    df_gemma = df
    df_train = df_gemma[df_gemma[ID_COL].isin(df_train_ref[ID_COL])].reset_index(drop=True)
    df_test  = df_gemma[df_gemma[ID_COL].isin(df_test_ref[ID_COL])].reset_index(drop=True)
    print(f"   Train: {df_train.shape} | Test: {df_test.shape}")
    df_train.to_csv('train_df_gemma.csv', index=False)
    df_test.to_csv('test_df_gemma.csv',  index=False)

else:
    print("File split belum ada -- melakukan GroupShuffleSplit baru...")
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df[GROUP_COL]))
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test  = df.iloc[test_idx].reset_index(drop=True)
    df_train.to_csv('train_df.csv', index=False)
    df_test.to_csv('test_df.csv',  index=False)

test_dataset = MultiTaskDataset(df_test, tokenizer, MAX_LEN, TEXT_TRAITS)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTotal: {len(df)} | Train: {len(df_train)} | Test: {len(df_test)}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

File split belum ada -- melakukan GroupShuffleSplit baru...

Total: 1054 | Train: 867 | Test: 187


## STAGE 2 — Grid Search + GroupKFold (MTL)

In [8]:
# Define fixed parameters. These are taken from the original param_grid's only combination.
params = {
    'learning_rate': 2e-5,
    'dropout_rate':  0.2,
    'epochs':        15,
}

# The following variables and functions were already defined outside the grid search loop
# and are necessary for the cross-validation.
gkf       = GroupKFold(n_splits=K_FOLDS)
# version-safe GradScaler (positional vs keyword `device` differs across torch versions)
def _make_scaler(dev_type):
    if dev_type not in ("cuda", "xpu"):
        return None
    try:
        return torch.amp.GradScaler(device=dev_type)
    except TypeError:
        try:
            return torch.amp.GradScaler(dev_type)
        except Exception:
            return torch.cuda.amp.GradScaler() if dev_type == "cuda" else None
scaler = _make_scaler(DEVICE.type)
criterion = nn.MSELoss()

# Display the parameters being used for this single run
print(f"\nStarting K-Fold Cross-Validation with fixed parameters:")
print(f"  LR={params['learning_rate']} | Drop={params['dropout_rate']} | Epochs={params['epochs']}")

fold_qwks_per_trait = {t: [] for t in TEXT_TRAITS}
fold_qwks_mean      = []

# K-Fold Cross-validation loop for the fixed parameters
for fold, (f_train_idx, f_val_idx) in enumerate(
        gkf.split(df_train, groups=df_train[GROUP_COL])):

    # Modified checkpoint path: removed 'gs_combo{i}'
    fold_ckpt = os.path.join(CHECKPOINT_DIR, f"fold{fold}.pth")

    # Modified seed: removed dependence on 'i' (combo index)
    current_seed = 42 + fold
    set_seed(current_seed)
    g = torch.Generator(); g.manual_seed(current_seed)

    f_train_df = df_train.iloc[f_train_idx]
    f_val_df   = df_train.iloc[f_val_idx]

    f_train_loader = DataLoader(
        MultiTaskDataset(f_train_df, tokenizer, MAX_LEN, TEXT_TRAITS),
        batch_size=BATCH_SIZE, shuffle=True, generator=g)
    f_val_loader = DataLoader(
        MultiTaskDataset(f_val_df, tokenizer, MAX_LEN, TEXT_TRAITS),
        batch_size=BATCH_SIZE, shuffle=False)

    model     = RobertaSemanticAES(MODEL_NAME, dropout_rate=params['dropout_rate']).to(DEVICE)
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=params['learning_rate'])

    start_epoch = 0
    if os.path.exists(fold_ckpt):
        start_epoch, _ = load_checkpoint(fold_ckpt, model, optimizer, scaler)
        start_epoch += 1

    for epoch in range(start_epoch, params['epochs']):
        model.train()
        for step, batch in enumerate(f_train_loader):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            bs   = batch['bertscore'].to(DEVICE).unsqueeze(1)
            tgt  = batch['targets'].to(DEVICE)

            if scaler:
                dtype = torch.bfloat16 if DEVICE.type == "xpu" else torch.float16
                with torch.autocast(device_type=DEVICE.type, dtype=dtype):
                    pred = model(input_ids=ids, attention_mask=mask, bertscore=bs)
                    loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                scaler.scale(loss).backward()
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(f_train_loader):
                    scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            else:
                pred = model(input_ids=ids, attention_mask=mask, bertscore=bs)
                loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                loss.backward()
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(f_train_loader):
                    optimizer.step(); optimizer.zero_grad()

            del ids, mask, bs, tgt, pred, loss

        _clear_gpu()
        save_checkpoint(fold_ckpt, epoch, model, optimizer, scaler)

    qwks = evaluate_qwk(model, f_val_loader)
    for t in TEXT_TRAITS:
        fold_qwks_per_trait[t].append(qwks[t])
    fold_qwks_mean.append(qwks['mean'])
    trait_str = " | ".join([f"{t}={qwks[t]:.4f}" for t in TEXT_TRAITS])
    print(f"   Fold {fold+1} QWK: mean={qwks['mean']:.4f} | {trait_str}")

    del model, optimizer
    _clear_gpu()

    if os.path.exists(fold_ckpt):
        os.remove(fold_ckpt)

mean_cv_qwk     = float(np.mean(fold_qwks_mean))
per_trait_means = {t: float(np.mean(fold_qwks_per_trait[t])) for t in TEXT_TRAITS}
print(f"Rata-rata Mean-QWK: {mean_cv_qwk:.4f} "
      f"({', '.join([f'{t}={per_trait_means[t]:.4f}' for t in TEXT_TRAITS])})")

print("\n" + "="*60)
print("CROSS-VALIDATION SELESAI!")
print(f"Final Parameters: {params}")
print("="*60)


Starting K-Fold Cross-Validation with fixed parameters:
  LR=2e-05 | Drop=0.2 | Epochs=15


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 1)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 2)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 3)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 4)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 5)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 6)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 7)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 8)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 9)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 10)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 11)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 12)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 13)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 14)
   Checkpoint tersimpan: checkpoints_unified/fold0.pth (epoch 15)
   Fold 1 QWK: mean

...

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 1)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 2)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 3)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 4)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 5)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 6)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 7)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 8)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 9)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 10)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 11)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 12)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 13)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 14)
   Checkpoint tersimpan: checkpoints_unified/fold1.pth (epoch 15)
   Fold 2 QWK: mean

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 1)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 2)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 3)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 4)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 5)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 6)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 7)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 8)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 9)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 10)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 11)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 12)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 13)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 14)
   Checkpoint tersimpan: checkpoints_unified/fold2.pth (epoch 15)
   Fold 3 QWK: mean

## STAGE 3 — Final Training (100% data train)

In [9]:
FINAL_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "final_training_latest.pth")

_clear_gpu()
set_seed(42)
g_final = torch.Generator(); g_final.manual_seed(42)

full_train_loader = DataLoader(
    MultiTaskDataset(df_train, tokenizer, MAX_LEN, TEXT_TRAITS),
    batch_size=BATCH_SIZE, shuffle=True, generator=g_final)

# User-defined parameters
final_dropout_rate  = 0.2
final_learning_rate = 2e-5
final_epochs        = 15

final_model     = RobertaSemanticAES(MODEL_NAME, dropout_rate=final_dropout_rate).to(DEVICE)
final_optimizer = AdamW(
    filter(lambda p: p.requires_grad, final_model.parameters()),
    lr=final_learning_rate)

start_epoch = 0
if os.path.exists(FINAL_CKPT_PATH):
    start_epoch, _ = load_checkpoint(FINAL_CKPT_PATH, final_model, final_optimizer, scaler)
    start_epoch += 1
    print(f"   Lanjut dari epoch {start_epoch + 1}/{final_epochs}")
else:
    print(f"   Mulai dari epoch 1/{final_epochs}")

if start_epoch >= final_epochs:
    print("Final training sudah selesai sebelumnya, melewati pelatihan ulang.")
else:
    for epoch in range(start_epoch, final_epochs):
        final_model.train()
        loop = tqdm(full_train_loader, desc=f"Final Epoch {epoch+1}/{final_epochs}")

        for step, batch in enumerate(loop):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            bs   = batch['bertscore'].to(DEVICE).unsqueeze(1)
            tgt  = batch['targets'].to(DEVICE)

            if scaler:
                dtype = torch.bfloat16 if DEVICE.type == "xpu" else torch.float16
                with torch.autocast(device_type=DEVICE.type, dtype=dtype):
                    pred = final_model(input_ids=ids, attention_mask=mask, bertscore=bs)
                    loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                scaler.scale(loss).backward()
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(full_train_loader):
                    scaler.step(final_optimizer); scaler.update(); final_optimizer.zero_grad()
            else:
                pred = final_model(input_ids=ids, attention_mask=mask, bertscore=bs)
                loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                loss.backward()
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(full_train_loader):
                    final_optimizer.step(); final_optimizer.zero_grad()

            loop.set_postfix({'Loss': f"{(loss.item() * ACCUMULATION_STEPS):.4f}"})

        save_checkpoint(FINAL_CKPT_PATH, epoch, final_model, final_optimizer, scaler)

    torch.save(final_model.state_dict(), BEST_MODEL_SAVE_PATH)
    print(f"Final Model tersimpan: {BEST_MODEL_SAVE_PATH}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   Mulai dari epoch 1/15


Final Epoch 1/15: 100%|██████████| 434/434 [00:23<00:00, 18.15it/s, Loss=0.5363]


   Checkpoint tersimpan: checkpoints_unified/final_training_latest.pth (epoch 1)


...

   Checkpoint tersimpan: checkpoints_unified/final_training_latest.pth (epoch 13)


Final Epoch 14/15: 100%|██████████| 434/434 [00:23<00:00, 18.23it/s, Loss=0.2920]


   Checkpoint tersimpan: checkpoints_unified/final_training_latest.pth (epoch 14)


Final Epoch 15/15: 100%|██████████| 434/434 [00:23<00:00, 18.28it/s, Loss=0.1351]


   Checkpoint tersimpan: checkpoints_unified/final_training_latest.pth (epoch 15)
Final Model tersimpan: best_roberta_unified_final_2.pth


## STAGE 4 — Extract Predictions ke CSV

In [10]:
train_csv_name = 'roberta_train_unified.csv'
test_csv_name  = 'roberta_test_unified.csv'

if os.path.exists(train_csv_name) and os.path.exists(test_csv_name):
    print("File prediksi sudah ada, melewati Stage 4.")
    df_train_out = pd.read_csv(train_csv_name)
    df_test_out  = pd.read_csv(test_csv_name)
    test_preds   = {t: df_test_out[f'prediction_{t}'].tolist()   for t in TEXT_TRAITS}
    test_targets = {t: df_test_out[f'ground_truth_{t}'].tolist() for t in TEXT_TRAITS}
else:
    if not os.path.exists(BEST_MODEL_SAVE_PATH):
        raise FileNotFoundError(
            f"Model {BEST_MODEL_SAVE_PATH} tidak ditemukan. "
            "Jalankan Stage 3 terlebih dahulu."
        )
    final_model = RobertaSemanticAES(MODEL_NAME, dropout_rate=final_dropout_rate).to(DEVICE)
    final_model.load_state_dict(torch.load(BEST_MODEL_SAVE_PATH, map_location=DEVICE))
    final_model.eval()

    train_extract_loader = DataLoader(
        MultiTaskDataset(df_train, tokenizer, MAX_LEN, TEXT_TRAITS),
        batch_size=BATCH_SIZE, shuffle=False)

    def extract_predictions(loader, desc):
        all_preds   = {t: [] for t in TEXT_TRAITS}
        all_targets = {t: [] for t in TEXT_TRAITS}
        with torch.no_grad():
            for batch in tqdm(loader, desc=desc):
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                bs   = batch['bertscore'].to(DEVICE).unsqueeze(1)
                tgt  = batch['targets'].numpy()

                if DEVICE.type in ["cuda", "xpu"]:
                    dtype = torch.bfloat16 if DEVICE.type == "xpu" else torch.float16
                    with torch.autocast(device_type=DEVICE.type, dtype=dtype):
                        preds = final_model(input_ids=ids, attention_mask=mask, bertscore=bs)
                else:
                    preds = final_model(input_ids=ids, attention_mask=mask, bertscore=bs)
                preds = preds.float().cpu().numpy()
                for ti, t in enumerate(TEXT_TRAITS):
                    all_preds[t].extend(preds[:, ti].flatten())
                    all_targets[t].extend(tgt[:, ti].flatten())
        return all_preds, all_targets

    train_preds, train_targets = extract_predictions(train_extract_loader, "Extracting Train Preds")
    test_preds,  test_targets  = extract_predictions(test_loader,          "Extracting Test Preds")

    train_cols, test_cols = {}, {}
    for t in TEXT_TRAITS:
        train_cols[f'ground_truth_{t}']       = train_targets[t]
        train_cols[f'prediction_{t}']         = train_preds[t]
        train_cols[f'rounded_prediction_{t}'] = np.round(np.array(train_preds[t]) * 2) / 2
        test_cols[f'ground_truth_{t}']        = test_targets[t]
        test_cols[f'prediction_{t}']          = test_preds[t]
        test_cols[f'rounded_prediction_{t}']  = np.round(np.array(test_preds[t]) * 2) / 2

    df_train_out = pd.DataFrame(train_cols)
    df_test_out  = pd.DataFrame(test_cols)

    df_train_out.to_csv(train_csv_name, index=False)
    df_test_out.to_csv(test_csv_name,  index=False)
    print(f"Train predictions -> {train_csv_name}")
    print(f"Test  predictions -> {test_csv_name}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Extracting Test Preds: 100%|██████████| 94/94 [00:01<00:00, 47.49it/s]

Train predictions -> roberta_train_unified.csv
Test  predictions -> roberta_test_unified.csv


## STAGE 5 — Evaluasi Akhir

In [11]:
final_qwks = {}
for t in TEXT_TRAITS:
    pred_classes = np.round(np.array(test_preds[t])   * 2).astype(int)
    true_classes = np.round(np.array(test_targets[t]) * 2).astype(int)
    final_qwks[t] = cohen_kappa_score(true_classes, pred_classes, weights='quadratic')

mean_qwk = float(np.mean(list(final_qwks.values())))
print("\n=== QWK FINAL (TEST SET) ===")
for t in TEXT_TRAITS:
    print(f"  {t:>12s} : {final_qwks[t]:.4f}")
print(f"  {'mean':>12s} : {mean_qwk:.4f}")


=== QWK FINAL (TEST SET) ===
  organizational_structure : 0.5062
     coherence : 0.5470
  essay_length : 0.3855
  grammatical_accuracy : 0.6356
  grammatical_diversity : 0.4933
  lexical_accuracy : 0.4915
  lexical_diversity : 0.5325
  punctuation_accuracy : 0.5237
  argument_clarity : 0.2055
  justifying_persuasiveness : 0.4452
          mean : 0.4766
